In [2]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path("/home/chupchik/Рабочий стол/fisrt_stepD/output")
parquet_path = OUT_DIR / "ait_ads_wazuh.parquet"

df_wazuh = pd.read_parquet(parquet_path)

print("Loaded:", df_wazuh.shape)
df_wazuh.head(3)

Loaded: (2600263, 14)


,file,timestamp,location,agent_id,agent_name,agent_ip,hostname,program,full_log,rule_id,rule_description,rule_groups,rule_level,rule_groups_str
0,fox_wazuh.json,2022-01-15 02:32:32+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:32 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
1,fox_wazuh.json,2022-01-15 02:32:32+00:00,/var/log/syslog,6,wazuh-client,192.168.128.170,taylorcruz-mail,freshclam,Jan 15 02:32:32 taylorcruz-mail freshclam[2851...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
2,fox_wazuh.json,2022-01-15 02:32:37+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:37 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"


In [3]:
import re

cols = [
    "timestamp","agent_ip","location","hostname","program",
    "rule_level","rule_id","rule_groups_str","rule_description","full_log"
]

sample = df_wazuh.copy()

# создадим y_priority если его нет
def map_priority(level):
    if pd.isna(level): return None
    level = float(level)
    if level <= 3: return "low"
    if level <= 6: return "medium"
    if level <= 10: return "high"
    return "critical"

sample["y_priority"] = sample["rule_level"].apply(map_priority)

sample = sample.dropna(subset=["full_log","timestamp"]).sample(30, random_state=42)[cols + ["y_priority"]].copy()

# Маскируем IP
def mask_ip(ip):
    if not isinstance(ip, str): return ip
    return re.sub(r"(\d+\.\d+\.\d+)\.\d+", r"\1.0", ip)

sample["agent_ip"] = sample["agent_ip"].apply(mask_ip)

# Укорачиваем лог
sample["full_log"] = sample["full_log"].astype(str).str.slice(0, 150)

sample

,timestamp,agent_ip,location,hostname,program,rule_level,rule_id,rule_groups_str,rule_description,full_log,y_priority
1623872,2022-01-30 07:43:52+00:00,192.168.2.0,/var/log/apache2/intranet-access.log,NaN,NaN,5,31101,"web,accesslog,attack",Web server 400 error code.,10.38.243.125 - - [30/Jan/2022:07:43:52 +0000]...,medium
1909172,2022-01-30 07:54:49+00:00,192.168.2.0,/var/log/apache2/intranet-access.log,NaN,NaN,5,31101,"web,accesslog,attack",Web server 400 error code.,10.38.243.125 - - [30/Jan/2022:07:54:49 +0000]...,medium
373916,2022-01-18 12:34:23+00:00,10.35.35.0,/var/log/apache2/intranet-access.log,NaN,NaN,10,31151,"web,accesslog,web_scan,recon",Multiple web server 400 error codes from same ...,172.17.130.196 - - [18/Jan/2022:12:34:23 +0000...,high
764,2022-01-15 08:39:06+00:00,172.17.131.0,/var/log/syslog,mail,dovecot,3,9701,"dovecot,authentication_success",Dovecot Authentication Success.,Jan 15 08:39:06 mail dovecot: imap-login: Logi...,low
1558901,2022-01-30 07:41:04+00:00,192.168.2.0,/var/log/apache2/intranet-access.log,NaN,NaN,5,31101,"web,accesslog,attack",Web server 400 error code.,10.38.243.125 - - [30/Jan/2022:07:41:04 +0000]...,medium
244061,2022-01-18 12:28:14+00:00,10.35.35.0,/var/log/apache2/intranet-access.log,NaN,NaN,5,31101,"web,accesslog,attack",Web server 400 error code.,172.17.130.196 - - [18/Jan/2022:12:28:14 +0000...,medium
271089,2022-01-18 12:29:31+00:00,10.35.35.0,/var/log/apache2/intranet-access.log,NaN,NaN,5,31101,"web,accesslog,attack",Web server 400 error code.,172.17.130.196 - - [18/Jan/2022:12:29:31 +0000...,medium
1670588,2022-01-30 07:45:41+00:00,192.168.2.0,/var/log/apache2/intranet-access.log,NaN,NaN,5,31101,"web,accesslog,attack",Web server 400 error code.,10.38.243.125 - - [30/Jan/2022:07:45:41 +0000]...,medium
434135,2022-01-18 12:37:12+00:00,10.35.35.0,/var/log/apache2/intranet-access.log,NaN,NaN,5,31101,"web,accesslog,attack",Web server 400 error code.,172.17.130.196 - - [18/Jan/2022:12:37:12 +0000...,medium
1289033,2022-01-19 11:13:23+00:00,192.168.96.0,/var/log/mail.info,watkins-mail,dovecot,3,9701,"dovecot,authentication_success",Dovecot Authentication Success.,Jan 19 11:13:23 watkins-mail dovecot: imap-log...,low
